# Gold - dependency-ordered runner

Plans, gates, builds and validates Gold tables in dependency order (dimensions before the facts that join them).
Source and target schemas are parameters: `cdr_mimiciv.silver` -> `cdr_mimiciv.gold` by default.

## Parameters
| Parameter | Default | Meaning |
|---|---|---|
| source_schema | cdr_mimiciv.silver | Silver `catalog.schema` |
| target_schema | cdr_mimiciv.gold | Gold `catalog.schema` |
| tables | all | `all` (preview only), comma-separated names, or a JSON list. A real run needs an explicit list |
| dry_run | true | Print plan, readiness axes, metadata checks and SQL; write nothing |
| publish_mode | create_or_replace | `insert_overwrite` keeps the existing table object but refuses any schema change |
| date_start / date_end | 2090-01-01 / 2250-12-31 | `dim_date` range |
| session_time_zone | UTC | Decides which calendar day a timestamp falls on (assumption: Bronze was read under UTC) |
| confirm_large_scan | false | Mandatory for tables flagged `scan=large` (prescriptions, emar, labevents, ...). Pass counts: `docs/gold_build_cost.md` |
| run_observe_checks | false | Large tables skip OBSERVE checks (one extra pass each) unless true |
| stop_on_failure | true | Stop at the first blocked or failed table |
| show_sql | true | Print each table's rendered `CREATE OR REPLACE TABLE` statement |

## What stops a real run
- `tables=all` (preview only).
- `approved_to_publish=False` or `silver_sources_confirmed=False` in `00_Gold_Registry` (every table, today).
- Any table flagged `scan=large` (no large build can run until a reviewed candidate-table validation exists; `confirm_large_scan` does not override this).
- A Silver source the local Silver registry marks `NEEDS_PROFILING` or `DEFERRED` (provider, services, labevents, microbiologyevents, chartevents). Live table state is not confirmed from local evidence.
- Silver tables/columns/types that differ from the contract (checked from metadata only).
- A joined dimension that is not unique on its business key, or a blocking pre-check.

Complete `docs/gold_governance_checks.md` before any real run. Nothing is dropped; there is no override parameter.

In [0]:
%run ./00_Gold_Registry

In [0]:
%run ./01_Gold_Engine

In [0]:
declare_widgets(dbutils)
cfg = read_config(dbutils)

In [0]:
results = run_gold(spark, cfg, group=None)